In [2]:
import polars as pl
from constant import openalex_path
from nltk.stem import PorterStemmer
import re
from sklearn.feature_extraction import text

import nltk
nltk.download("punkt")

stemmer = PorterStemmer()

def tokenizer(doc, custom_stopwords=None):
    default_sw_raw = text.ENGLISH_STOP_WORDS
    default_sw_stemmed = {stemmer.stem(w) for w in default_sw_raw}

    if custom_stopwords:
        custom_sw_stemmed = {stemmer.stem(w.lower()) for w in custom_stopwords}
    else:
        custom_sw_stemmed = set()

    stop_words = default_sw_stemmed | custom_sw_stemmed
    tokens = re.findall(r"[A-Za-z]+", doc)
    return [stemmer.stem(t.lower()) for t in tokens
            if stemmer.stem(t.lower()) not in stop_words]

[nltk_data] Downloading package punkt to /home/ec2-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
CNS_papers_details_df = pl.read_parquet("./data/CNS_papers_inferred_subjects_Life.parquet")
CNS_citing_preprints_df = pl.read_parquet("./data/citing_preprints_full.parquet").filter(pl.col("work_id").is_in(CNS_papers_details_df["work_id"].to_list())).lazy()
journal_detail = pl.read_parquet("./data/CNS_journal_inception.parquet")
title_lf = pl.scan_parquet(openalex_path + "/works/display_name/*").select(
    pl.col("work_id"),
    pl.col("display_name").alias("text")
)
abstract_lf = pl.scan_parquet(openalex_path + "/works/abstract/*").select(
    pl.col("work_id"),
    pl.col("abstract").alias("text")
)
ref_lf =pl.scan_parquet(openalex_path + "/works/referenced_works/*")
publication_year_lf = pl.scan_parquet(openalex_path + "/works/publication_year/*")

preprint = pl.read_parquet("./data/preprints_opal.parquet").drop_nulls("work_id")

In [ ]:
# ref_detail = (
#     ref_lf
#     .join(CNS_citing_preprints_df, on="work_id", how="semi")
#     .explode("referenced_works")
#     .join(abstract_lf, left_on="referenced_works", right_on="work_id", how="left")
#     # .join(title_lf, left_on="referenced_works", right_on="work_id", how="left")
#     .join(preprint
#           .select(
#               pl.col("work_id"),
#               pl.col("curated_doi").is_not_null().alias("is_curated"),
#               pl.col("server")
#             ).lazy(),
#             left_on="referenced_works",
#             right_on="work_id",
#             how="left"
#     )
# )

In [ ]:
# CNS_references_detail = (
#     ref_detail
#     .join(
#         CNS_papers_details_df
#         .lazy().join(journal_detail.lazy(), on="primary_location_id", how="left"),
#         on="work_id",
#         how="left"
#     )
#     .with_columns(
#         pl.col("text").map_elements(tokenizer, return_dtype=pl.List(pl.Utf8)).alias("tokens")
#     )
# )

# CNS_references_detail.sink_parquet("./data/CNS_references_details_abstract.parquet")
# # CNS_references_detail.write_parquet("./data/CNS_references_details_abstract.parquet")

/tmp/ipykernel_103537/4266027141.py:14: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  CNS_references_detail.sink_parquet("./data/CNS_references_details_abstract.parquet")


# Curation venues

In [ ]:
doi_lf = pl.scan_parquet(openalex_path + "/works/doi/*").with_columns(
    pl.col("doi").str.to_lowercase().str.replace("^https://doi.org/", "")
)
source_lf = pl.scan_parquet(openalex_path + "/works/primary_location/*").select(
preprint.

doi,primary_category,server,curated_doi,published_date,work_id,primary_location_id,publication_year
str,str,str,str,datetime[μs],str,str,i64
"""10.31234/osf.io/u9wns_v1""","""Psychiatry""","""psyarxiv""",null,2024-06-11 00:00:00,"""W4407591363""",null,2024
"""10.31234/osf.io/u9wns_v1""","""Psychiatry""","""psyarxiv""",null,2024-06-11 00:00:00,"""W4407591363""",null,2024
"""10.31234/osf.io/u9wns_v1""","""Psychiatry""","""psyarxiv""",null,2024-06-11 00:00:00,"""W4407591363""",null,2024
"""10.31234/osf.io/g3dnz_v1""","""Social and Behavioral Sciences""","""psyarxiv""",null,2024-12-08 00:00:00,"""W4407750529""",null,2024
"""10.31234/osf.io/g3dnz_v1""","""Social and Behavioral Sciences""","""psyarxiv""",null,2024-12-08 00:00:00,"""W4407750529""",null,2024
…,…,…,…,…,…,…,…
"""10.48550/arxiv.2405.09344""","""cs.NI""","""arxiv""",null,2024-05-15 13:50:37,"""W4396987280""","""S4306400194""",2024
"""10.48550/arxiv.1909.12716""","""cs.SE""","""arxiv""",null,2019-09-27 14:40:42,"""W4288102205""","""S4306400194""",2019
"""10.48550/arxiv.1811.05353""","""math.NA""","""arxiv""",null,2018-11-13 00:00:00,"""W4289293270""","""S4306400194""",2018


# Representative topics by LLR

In [43]:
CNS_references_detail = pl.read_parquet("./data/CNS_references_details.parquet")

In [ ]:
from collections import Counter
import numpy as np

def extract_llr_keywords(tk_target, tk_bg):
    # count
    c_t = Counter(tk_target)
    c_b = Counter(tk_bg)

    vocab = np.array(list(c_t.keys() | c_b.keys()))
    
    # vectorize counts
    k11 = np.array([c_t[w] for w in vocab], dtype=float)   # target
    k21 = np.array([c_b[w] for w in vocab], dtype=float)   # background
    target_total = k11.sum()
    bg_total     = k21.sum()

    k12 = target_total - k11
    k22 = bg_total - k21

    # -----------------------------
    # vectorized log-likelihood
    # -----------------------------
    def _h(a, b):
        # safe log (avoid zero)
        return np.where(a == 0, 0.0, a * np.log(a / b))

    total_all = target_total + bg_total

    llr_vals = 2 * (
        _h(k11, k11 + k12) +
        _h(k12, k11 + k12) +
        _h(k21, k21 + k22) +
        _h(k22, k21 + k22) -
        _h(k11 + k21, total_all) -
        _h(k12 + k22, total_all)
    )

    E11 = (k11 + k21) * target_total / total_all
    E21 = (k11 + k21) * bg_total     / total_all

    p_target = k11 / target_total
    p_bg   = k21 / bg_total

    # np array of 1.0 or -1.0
    sign = np.where(p_target > p_bg, 1.0, -1.0)

    # ソート
    order = np.argsort(-llr_vals)   # descending
    return {"vocab": vocab[order], "llr": llr_vals[order], "E11": E11[order], "E21": E21[order], "sign": sign[order]}


# --- Example ---
target = ("We propose a novel citation-based graph [SEP] model for scientific [SEP] knowledge.").split()
corpus = """All previous studies in scientometrics have built citation graphs 
using co-citation and [SEP] bibliographic coupling, but few models incorporate 
developmental edges or push-citation [SEP] mechanisms.""".split()

pl.DataFrame(extract_llr_keywords(target, corpus))  # top 20

/tmp/ipykernel_15322/517086185.py:25: RuntimeWarning: divide by zero encountered in log
  return np.where(a == 0, 0.0, a * np.log(a / b))
/tmp/ipykernel_15322/517086185.py:25: RuntimeWarning: invalid value encountered in multiply
  return np.where(a == 0, 0.0, a * np.log(a / b))


vocab,llr,E11,E21,sign
str,f64,f64,f64,f64
"""We""",2.310498,0.324324,0.675676,1.0
"""scientific""",2.310498,0.324324,0.675676,1.0
"""citation-based""",2.310498,0.324324,0.675676,1.0
"""knowledge.""",2.310498,0.324324,0.675676,1.0
"""for""",2.310498,0.324324,0.675676,1.0
…,…,…,…,…
"""coupling,""",0.797355,0.324324,0.675676,-1.0
"""All""",0.797355,0.324324,0.675676,-1.0
"""have""",0.797355,0.324324,0.675676,-1.0


In [22]:
from collections import Counter
import numpy as np

def llr_for_lexicon(tk_target, tk_bg, lexicon, *, return_p=False):
    """
    Test whether a given lexicon (list[str]) is enriched in tk_target vs tk_bg
    using log-likelihood ratio (G-test) on a 2x2 contingency table:

                 in_lex   not_in_lex
    target         k11       k12
    background     k21       k22

    Null: lexicon frequency is the same in target and background.
    """
    c_t = Counter(tk_target)
    c_b = Counter(tk_bg)

    lex = set(lexicon)

    # observed counts (sum over words in lexicon)
    k11 = float(sum(c_t[w] for w in lex))  # target, in_lex
    k21 = float(sum(c_b[w] for w in lex))  # bg,     in_lex

    target_total = float(sum(c_t.values()))
    bg_total     = float(sum(c_b.values()))

    k12 = target_total - k11               # target, not_in_lex
    k22 = bg_total - k21                   # bg,     not_in_lex

    # sanity (should be non-negative)
    if min(k11, k12, k21, k22) < 0:
        raise ValueError("Negative cell count: check totals / inputs.")

    # vectorized safe term a*log(a/b)
    def _h(a, b):
        return 0.0 if a == 0.0 else a * np.log(a / b)

    total_all = target_total + bg_total

    # LLR (G-test) for 2x2
    llr = 2.0 * (
        _h(k11, k11 + k12) +
        _h(k12, k11 + k12) +
        _h(k21, k21 + k22) +
        _h(k22, k21 + k22) -
        _h(k11 + k21, total_all) -
        _h(k12 + k22, total_all)
    )

    # expected counts under null
    E11 = (k11 + k21) * target_total / total_all
    E21 = (k11 + k21) * bg_total     / total_all

    # direction (enriched in target?)
    p_target = 0.0 if target_total == 0 else k11 / target_total
    p_bg     = 0.0 if bg_total == 0 else k21 / bg_total
    sign = 1.0 if p_target > p_bg else (-1.0 if p_target < p_bg else 0.0)

    out = {
        "llr": llr,
        "sign": sign,
        "k11": k11, "k12": k12, "k21": k21, "k22": k22,
        "E11": E11, "E21": E21,
        "prob_target": p_target, "prob_bg": p_bg,
        "lexicon_size": len(lex),
    }

    if return_p:
        # chi-square approximation with df=1: p = P(ChiSq_1 >= llr)
        try:
            from scipy.stats import chi2
            out["p_value_chi2_df1"] = float(chi2.sf(llr, df=1))
        except Exception:
            out["p_value_chi2_df1"] = None

    return out


In [9]:
(
    CNS_references_detail
    .filter(
        pl.col("is_curated")==False,
        pl.col("server").is_not_null(),
    )
    .select("tokens", "publication_year")
    .explode("tokens")
    .group_by("tokens").len()
    .sort("len", descending=True)
    .filter(pl.col("len") > 10)
    .write_csv("./data/CNS_references_detail_tokens.csv")
)

In [58]:
year = (2013, 2024)

corpus_target = (
    CNS_references_detail
    .unique(pl.col("referenced_works"))
    .filter(
        pl.col("publication_year").is_between(year[0], year[1]),
        pl.col("server").is_in(["medrxiv", "biorxiv"]),
        pl.col("is_curated")==False, # curated
        pl.col("server").is_not_null() # preprint
    )
    .get_column("tokens").explode().to_list() # text is a list column
)

corpus_bg = (
    CNS_references_detail
    .unique(pl.col("referenced_works"))
    .filter(
        pl.col("publication_year").is_between(year[0], year[1]),
        pl.col("server").is_in(["medrxiv", "biorxiv"]),
        pl.col("is_curated")==True, # curated
        pl.col("server").is_not_null() # preprint
    )
    .get_column("tokens").explode().to_list() # text is a list column
)

len(corpus_target), len(corpus_bg)

(49290, 69665)

In [57]:
keywords = pl.DataFrame(extract_llr_keywords(corpus_target, corpus_bg))

filtered = (
    keywords
    .filter(
        # chi^2 <1% 
        pl.col("llr") >= 6.635,
        pl.col("E11") >= 11,
        pl.col("E21") >= 11
    )
)

print(
    filtered
    .filter(pl.col("sign")>0)
    .get_column("vocab")
    .to_list()
)
print(
    filtered
    .filter(pl.col("sign")<0)
    .get_column("vocab")
    .to_list()
)
filtered

['learn', 'network', 'convolut', 'deep', 'adversari', 'neural', 'machin', 'train', 'reinforc', 'languag', 'model', 'task', 'algorithm', 'approach', 'supervis', 'graph', 'novel', 'optim', 'patholog', 'cognit', 'understand', 'imag', 'gener', 'recurr', 'use', 'rang', 'probabilist', 'object', 'alzheim', 'bayesian', 's', 'infer', 'explain', 'ai', 'person', 'attent', 'multimod', 'interpret', 'segment', 'al', 'u', 'imput', 'inform']
['cov', 'sar', 'covid', 'patient', 'basi', 'serolog', 'infect', 'cryo', 'contact', 'bacteri', 'antibodi', 'parasit', 'assembl', 'loop', 'protein', 'structur', 'bnt', 'pandem', 'genom', 'transcript', 'vaccin', 'particl', 'phase', 'receptor', 'rapid', 'termin', 'decis', 'european', 'impact', 'crispr', 'divers']


/tmp/ipykernel_15322/517086185.py:25: RuntimeWarning: divide by zero encountered in log
  return np.where(a == 0, 0.0, a * np.log(a / b))
/tmp/ipykernel_15322/517086185.py:25: RuntimeWarning: invalid value encountered in multiply
  return np.where(a == 0, 0.0, a * np.log(a / b))


vocab,llr,E11,E21,sign
str,f64,f64,f64,f64
"""learn""",132.37223,270.666417,328.333583,1.0
"""network""",77.53037,228.643084,277.356916,1.0
"""cov""",67.402198,848.148356,1028.851644,-1.0
"""sar""",64.061544,840.014808,1018.985192,-1.0
"""convolut""",53.581255,37.052832,44.947168,1.0
…,…,…,…,…
"""european""",6.891601,12.652186,15.347814,-1.0
"""impact""",6.869157,79.076165,95.923835,-1.0
"""inform""",6.773198,56.482975,68.517025,1.0


In [28]:
dictionary_datapaper = [
    "dataset", "dataset", "resource", "repository", "compendium", "atlas", "catalog", "census", "survey",
    "atlas", "map", "landscape", "profiling", "characterization", "systematic", "comprehensive", "single-cell", "multi-omics",
    "proteome", "metabolome", "interactome", "connectome", "phenome",
    "benchmark", "reference", "annotation", "curated", "harmonized", "standardized",
]
dictionary_codepaper = [
    "tool", "software", "package", "library", "framework", "platform", "pipeline", "workflow", "suite",
    "algorithm", "method", "model", "inference", "prediction", "classification", "clustering", "embedding",
    "scalable", "efficient", "fast", "robust", "reproducible", "automated", "modular",
    "genomics", "transcriptomics", "proteomics", "imaging", "single-cell", "spatial", "multi-modal",
    "open-source", "API", "command-line", "web-based", "interactive"
]

year = (2019, 2024)

corpus_target = (
    CNS_references_detail
    .unique(pl.col("referenced_works"))
    .filter(
        pl.col("publication_year").is_between(year[0], year[1]),
        # pl.col("server") == "biorxiv",
        pl.col("is_curated")==False, # curated
        pl.col("server").is_not_null() # preprint
    )
    .get_column("tokens").explode().to_list() # text is a list column
)

corpus_bg = (
    CNS_references_detail
    .unique(pl.col("referenced_works"))
    .filter(
        pl.col("publication_year").is_between(year[0], year[1]),
        # pl.col("server") == "biorxiv",
        pl.col("is_curated")==True, # curated
        pl.col("server").is_not_null() # preprint
    )
    .get_column("tokens").explode().to_list() # text is a list column
)

print(llr_for_lexicon(
    corpus_target,
    corpus_bg,
    tokenizer(" ".join(dictionary_datapaper)),
    return_p=True
))
print(llr_for_lexicon(
    corpus_target,
    corpus_bg,
    tokenizer(" ".join(dictionary_codepaper)),
    return_p=True
))


{'llr': np.float64(0.5110604958899785), 'sign': -1.0, 'k11': 2118.0, 'k12': 52577.0, 'k21': 2539.0, 'k22': 61690.0, 'E11': 2141.8268389896066, 'E21': 2515.1731610103934, 'prob_target': 0.03872383216016089, 'prob_bg': 0.03953043017951393, 'lexicon_size': 29, 'p_value_chi2_df1': 0.47468013556606625}
{'llr': np.float64(17.805206785435075), 'sign': 1.0, 'k11': 3959.0, 'k12': 50736.0, 'k21': 4249.0, 'k22': 59980.0, 'E11': 3774.9870505532945, 'E21': 4433.0129494467055, 'prob_target': 0.07238321601608923, 'prob_bg': 0.06615391801211291, 'lexicon_size': 41, 'p_value_chi2_df1': 2.447126614872349e-05}


In [18]:
CNS_references_detail.filter(pl.col('server').is_not_null()).unique('referenced_works').explode('tokens').filter(pl.col('tokens')=='dataset')

work_id,referenced_works,text,is_curated,server,doi,inferred_subfield,inferred_field,primary_location_id,publication_year,ref_count,journal_inception,inception_period,tokens
str,str,str,bool,str,str,str,str,str,i64,u32,i64,cat,str
"""W4200613736""","""W2955323014""","""A large single-participant fMR…",false,"""biorxiv""","""10.1038/s41593-021-00962-x""","""Biological Sciences""","""Life Sciences and Medicine""","""S2298632""",2021,82,1998,"""Before 2000""","""dataset"""
"""W2962898942""","""W2791837701""","""Reproducible integration of mu…",true,"""biorxiv""","""10.1038/s41467-019-10945-z""","""Medicine""","""Life Sciences and Medicine""","""S64187185""",2019,73,2010,"""2000-2015""","""dataset"""
"""W4317885418""","""W4293873702""","""RadialGAN: Leveraging multiple…",false,"""arxiv""","""10.1016/j.heliyon.2023.e13200""","""Nursing""","""Life Sciences and Medicine""","""S2898612692""",2023,36,2015,"""2000-2015""","""dataset"""
"""W4308885736""","""W4297697565""","""UCF101: A Dataset of 101 Human…",false,"""arxiv""","""10.1038/s41467-022-34591-0""","""Medicine""","""Life Sciences and Medicine""","""S64187185""",2022,23,2010,"""2000-2015""","""dataset"""
"""W2943986726""","""W3103167480""","""The HCP 7T Retinotopy Dataset:…",false,"""biorxiv""","""10.1016/j.cub.2019.04.012""","""Medicine""","""Life Sciences and Medicine""","""S128425624""",2019,27,1991,"""Before 2000""","""dataset"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""W4307716642""","""W3212553536""","""The Immune Signatures Data Res…",true,"""biorxiv""","""10.1038/s41590-022-01329-5""","""Medicine""","""Life Sciences and Medicine""","""S41063453""",2022,41,2000,"""Before 2000""","""dataset"""
"""W4229370737""","""W3190504389""","""An Open MRI Dataset for Multis…",true,"""biorxiv""","""10.1038/s41467-022-29886-1""","""Medicine""","""Life Sciences and Medicine""","""S64187185""",2022,132,2010,"""2000-2015""","""dataset"""
"""W3175536848""","""W3109330824""","""Survival analysis in breast ca…",true,"""medrxiv""","""10.1016/j.celrep.2021.109285""","""Biological Sciences""","""Life Sciences and Medicine""","""S169236886""",2021,106,2012,"""2000-2015""","""dataset"""


In [179]:
CNS_references_detail.filter(pl.col('server').is_not_null()).unique('referenced_works').explode('tokens').filter(pl.col('tokens')=='genom').write_csv('./data/genom.csv')

In [ ]:
biorxiv only:
[+]
[-]

2013-2018:
[]
['genom']

2019-2021:
['novel', 'cancer']
['basi', 'cryo', 'structur', 'bacteri', 'protein', 'em', 'receptor', 'assembl']

2022-2024:
['patholog', 'fast']
['tissu']

2019-2022:
['rang', 'novel', 'tumor', 'vaccin', 'tool', 'diseas']
['basi', 'structur', 'protein', 'cryo', 'bacteri', 'translat', 'format']

2023-2024:
['deep']
['enhanc']

In [59]:
# biorxiv and medrxiv
set1 = set(['learn', 'network', 'convolut', 'deep', 'adversari', 'neural', 'machin', 'train', 'reinforc', 'languag', 'model', 'task', 'algorithm', 'approach', 'supervis', 'graph', 'novel', 'optim', 'patholog', 'cognit', 'understand', 'imag', 'gener', 'recurr', 'use', 'rang', 'probabilist', 'object', 'alzheim', 'bayesian', 's', 'infer', 'explain', 'ai', 'person', 'attent', 'multimod', 'interpret', 'segment', 'al', 'u', 'imput', 'inform'])
set2 = set(['cov', 'sar', 'covid', 'patient', 'basi', 'serolog', 'infect', 'cryo', 'contact', 'bacteri', 'antibodi', 'parasit', 'assembl', 'loop', 'protein', 'structur', 'bnt', 'pandem', 'genom', 'transcript', 'vaccin', 'particl', 'phase', 'receptor', 'rapid', 'termin', 'decis', 'european', 'impact', 'crispr', 'divers'])

# all
set1_ = set(['learn', 'network', 'convolut', 'deep', 'adversari', 'neural', 'machin', 'train', 'reinforc', 'languag', 'model', 'task', 'algorithm', 'approach', 'supervis', 'graph', 'novel', 'optim', 'patholog', 'cognit', 'understand', 'imag', 'gener', 'recurr', 'use', 'rang', 'probabilist', 'object', 'alzheim', 'bayesian', 's', 'infer', 'explain', 'ai', 'person', 'attent', 'multimod', 'interpret', 'segment', 'al', 'u', 'imput', 'inform'])
set2_ = set(['cov', 'sar', 'covid', 'patient', 'basi', 'serolog', 'infect', 'cryo', 'contact', 'bacteri', 'antibodi', 'parasit', 'assembl', 'loop', 'protein', 'structur', 'bnt', 'pandem', 'genom', 'transcript', 'vaccin', 'particl', 'phase', 'receptor', 'rapid', 'termin', 'decis', 'european', 'impact', 'crispr', 'divers'])

(set1 | set1_) - (set1 & set1_)

set()

In [60]:
(set2 | set2_) - (set2 & set2_)

set()

# difference in server count

In [63]:
CNS_references_detail

work_id,referenced_works,text,is_curated,server,doi,inferred_subfield,inferred_field,primary_location_id,publication_year,ref_count,journal_inception,inception_period,tokens
str,str,str,bool,str,str,str,str,str,i64,u32,i64,cat,list[str]
"""W3136578706""","""W3107501485""","""DNA polymerase theta suppresse…",true,"""biorxiv""","""10.1016/j.cub.2021.03.008""","""Biological Sciences""","""Life Sciences and Medicine""","""S128425624""",2021,44,1991,"""Before 2000""","[""dna"", ""polymeras"", … ""cross""]"
"""W3172218068""","""W3023771398""","""Notch1 switches progenitor com…",true,"""biorxiv""","""10.1016/j.celrep.2021.109208""","""Biological Sciences""","""Life Sciences and Medicine""","""S169236886""",2021,80,2012,"""2000-2015""","[""notch"", ""switch"", … ""medulloblastoma""]"
"""W4210636419""","""W3136946877""","""Bacterial-type plant ferroxida…",false,"""biorxiv""","""10.1038/s41467-022-28261-4""","""Agriculture and Forestry""","""Life Sciences and Medicine""","""S64187185""",2022,67,2010,"""2000-2015""","[""bacteri"", ""type"", … ""develop""]"
"""W3213101281""","""W3152562958""","""SENSV: Detecting Structural Va…",true,"""biorxiv""","""10.1038/s41587-021-01108-x""","""Biological Sciences""","""Life Sciences and Medicine""","""S106963461""",2021,362,1983,"""Before 2000""","[""sensv"", ""detect"", … ""flowcel""]"
"""W4200478178""","""W3196642639""","""Structure and RNA template req…",true,"""biorxiv""","""10.1126/science.abj9184""","""Biological Sciences""","""Life Sciences and Medicine""","""S3880285""",2021,65,1880,"""Before 2000""","[""structur"", ""rna"", … ""polymeras""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""W4304957710""","""W4220874692""",null,null,null,"""10.1038/s41467-022-33740-9""","""Medicine""","""Life Sciences and Medicine""","""S64187185""",2022,27,2010,"""2000-2015""",null
"""W3170813319""","""W3099343795""",null,null,null,"""10.1038/s41467-021-23783-9""","""Biological Sciences""","""Life Sciences and Medicine""","""S64187185""",2021,56,2010,"""2000-2015""",null
"""W3114941018""","""W2192783909""",null,null,null,"""10.1038/s41467-020-20157-5""","""Biological Sciences""","""Life Sciences and Medicine""","""S64187185""",2020,72,2010,"""2000-2015""",null


In [64]:
source_server_distribution = (
    CNS_references_detail
    .filter(pl.col('server').is_not_null()).unique('referenced_works')
    .group_by("is_curated","server").len()
    .pivot(
        on="is_curated",
        index="server",
    )
)

In [68]:
source_server_distribution.with_columns(
    ((pl.col("false") / pl.sum("false") * 100)).alias("proportion_uncurated"),
    ((pl.col("true") / pl.sum("true") * 100)).alias("proportion_curated")
)

server,false,true,proportion_uncurated,proportion_curated
str,u32,u32,f64,f64
"""psyarxiv""",183,54,2.965484,0.754612
"""osf""",53,15,0.858856,0.209614
"""biorxiv""",4208,5643,68.189921,78.856903
"""socarxiv""",20,2,0.324097,0.027949
"""arxiv""",960,12,15.556636,0.167691
"""medrxiv""",747,1430,12.105007,19.983231


In [51]:
term = (
    CNS_references_detail
    .filter(pl.col('server').is_not_null()).unique('referenced_works')
    .group_by("is_curated").agg(pl.col("server"))
    .sort("is_curated")
    .get_column("server")
    .to_list()
)

In [53]:
keywords = pl.DataFrame(extract_llr_keywords(term[0], term[1]))
keywords

vocab,llr,E11,E21,sign
str,f64,f64,f64,f64
"""arxiv""",1448.965545,450.079688,521.920312,1.0
"""biorxiv""",195.335353,4561.455766,5289.544234,-1.0
"""medrxiv""",153.254939,1008.048848,1168.951152,-1.0
"""psyarxiv""",96.225243,109.741652,127.258348,1.0
"""osf""",28.64774,31.487056,36.512944,1.0
"""socarxiv""",19.909733,10.186989,11.813011,1.0


# abstract tf-idf

In [ ]:
1-gram
curated
cell: 0.2798
use: 0.2123
sar: 0.2040
cov: 0.1951
data: 0.1640
infect: 0.1551
genom: 0.1489
protein: 0.1344
human: 0.1344
s: 0.1303
omicron: 0.1296
variant: 0.1192
gene: 0.1186
identifi: 0.1082
result: 0.0979
function: 0.0951
specif: 0.0930
associ: 0.0924
model: 0.0917
sequenc: 0.0903

non-curated 
model: 0.4151
use: 0.2144
data: 0.1867
function: 0.1749
cell: 0.1613
learn: 0.1482
sequenc: 0.1415
perform: 0.1368
structur: 0.1301
algorithm: 0.1236
reml: 0.1145
lmer: 0.1145
paramet: 0.1054
result: 0.1026
align: 0.1020
includ: 0.0987
variant: 0.0979
r: 0.0949
profil: 0.0932
method: 0.0928

In [ ]:
1-2gram
curated 
model: 0.3751
use: 0.1938
data: 0.1687
function: 0.1581
cell: 0.1458
learn: 0.1339
sequenc: 0.1279
perform: 0.1236
structur: 0.1176
algorithm: 0.1117
lmer: 0.1035
reml: 0.1035
paramet: 0.0953
result: 0.0927
align: 0.0921
includ: 0.0892
variant: 0.0885
r: 0.0857
profil: 0.0843

non-curated
model: 0.3751
use: 0.1938
data: 0.1687
function: 0.1581
cell: 0.1458
learn: 0.1339
sequenc: 0.1279
perform: 0.1236
structur: 0.1176
algorithm: 0.1117
lmer: 0.1035
reml: 0.1035
paramet: 0.0953
result: 0.0927
align: 0.0921
includ: 0.0892
variant: 0.0885
r: 0.0857
profil: 0.0843

In [ ]:
1-2gram, mutually exclusive

curated 
cell: 0.2661
use: 0.2019
sar: 0.1940
cov: 0.1855
sar cov: 0.1789
data: 0.1560
infect: 0.1475
genom: 0.1416
protein: 0.1278
human: 0.1278
s: 0.1239
omicron: 0.1232
variant: 0.1134
gene: 0.1127
identifi: 0.1029
result: 0.0931
function: 0.0904
specif: 0.0885
associ: 0.0878
model: 0.0872

non-curated 
model: 0.3751
use: 0.1938
data: 0.1687
function: 0.1581
cell: 0.1458
learn: 0.1339
sequenc: 0.1279
perform: 0.1236
structur: 0.1176
algorithm: 0.1117
lmer: 0.1035
reml: 0.1035
paramet: 0.0953
result: 0.0927
align: 0.0921
includ: 0.0892
variant: 0.0885
r: 0.0857
profil: 0.0843
method: 0.0839
base: 0.0824
effect: 0.0817
bwa: 0.0810
optim: 0.0786